### Atividade Busca Semântica

Nesta atividade você deve aplicar os conhecimentos sobre word embeddings e tokenização para criar um mecanismos de busca semântica. Será disponibilizado um conjunto de dados que possui perguntas médicas, em inglês. Esse conjunto de dados possui as perguntas e também as respostas.

Você deve gerar os vetores das perguntas do conjunto de dados, e permitir que o "usuário" envie a sua pergunta. Você também deve gerar o vetor da pergunta do usuário e com isso buscar a resposta ideal para o usuário. **A resposta ideal é aquela onde o vetor da pergunta do usuário é mais similar ao vetor da pergunta do conjunto de dados** Consulte o notebook "nlp2.ipynb" para verificar como realizamos esse processo

Portanto, no seu script deve ser possível escrever um texto "pergunta" e deve ser retornado a resposta adequada, isto é, a resposta associada a pergunta mais similar no conjunto de dados.

Para isso utilize o pandas e os packages do huggingface

In [5]:
### Download do conjunto de dados

import kagglehub

# Download latest version
path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")

print("Path to dataset files:", path)

100%|██████████| 4.95M/4.95M [00:01<00:00, 4.79MB/s]

Extracting files...


Path to dataset files: C:\Users\joao1\.cache\kagglehub\datasets\pythonafroz\medquad-medical-question-answer-for-ai-research\versions\1


In [6]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers.tokenization_utils_base import BatchEncoding

In [7]:
## Amostra do conjunto

df = pd.read_csv(f"{path}/medquad.csv")
df_amostra = df.sample(5000)

In [8]:
# Modelo para a língua inglesa

nome_modelo = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
model = AutoModel.from_pretrained(nome_modelo)

c:\Users\joao1\OneDrive\Documentos\SistemasDeInformacao8Periodo\IA\ml-supervised-dev\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joao1\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' packag

In [9]:
## Funções para: 1) Obter os tokens; 2) Obter os embeddings

def get_tokens(pergunta: str) -> str:
    return tokenizer(pergunta, return_tensors="pt")

def get_vetores(tokens_pergunta: BatchEncoding) -> Tensor:

    with torch.no_grad():
        outputs = model(**tokens_pergunta)
        embeddings = outputs.last_hidden_state

    return embeddings

In [10]:
## Novas colunas para os tokens e para os embeddings

df_amostra['tokens'] = df_amostra['question'].apply(lambda x: get_tokens(x))
df_amostra['vetores'] = df_amostra['tokens'].apply(lambda x: get_vetores(x))

In [11]:
## Busca semântica: encontre a resposta ideal para a pergunta do usuário
import torch.nn.functional as F
pergunta_usuario = input("Digite sua pergunta em inglês: ")
tokens_usuario = get_tokens(pergunta_usuario)
vetor_usuario = get_vetores(tokens_usuario)
def get_embedding_medio(vetores):
    # Calcula a média dos embeddings (ignora [CLS] e [SEP] se necessário)
    return vetores.mean(dim=1).squeeze(0)
embedding_usuario = get_embedding_medio(vetor_usuario)
embeddings_dataset = df_amostra['vetores'].apply(get_embedding_medio)
similaridades = embeddings_dataset.apply(lambda x: F.cosine_similarity(embedding_usuario, x, dim=0))
indice_mais_similar = similaridades.idxmax()
print("Pergunta mais similar:", df_amostra.loc[indice_mais_similar, 'question'])
print("Resposta ideal:", df_amostra.loc[indice_mais_similar, 'answer'])

Pergunta mais similar: Who is at risk for Diabetes? ?
Resposta ideal: Diabetes is a serious, life-long disease. It can lead to problems such as heart disease, stroke, vision loss, kidney disease, and nerve damage. More than 8 million people in the United States have type 2 diabetes and dont know it. Many people dont find out they have diabetes until they are faced with problems such as blurry vision or heart trouble. Certain factors can increase your risk for diabetes, and its important to know what they are. Type 1 Diabetes Type 1 diabetes is an autoimmune disease. In an autoimmune reaction, antibodies, or immune cells, attach to the bodys own healthy tissues by mistake, signaling the body to attack them. At present, scientists do not know exactly what causes the body's immune system to attack the cells, but many believe that both genetic factors and environmental factors, such as viruses, are involved. Studies are now underway to identify these factors and prevent type 1 diabetes in 